# Forward Pass Trajectory Animation

Visualises how a single input progresses through the frozen backbone and
meta-encoder simultaneously:

| Panel | Content |
|-------|---------|
| **Top-left** | Raw input image (static) |
| **Top-right** | Predicted vs. true class label (static) |
| **Left** | 3-D stacked activation heatmaps — one plane per backbone block. Layer runs left (L1) to right (L8). Current layer is bright; past layers fade; future layers are ghost-dim. |
| **Right** | Circuit flow diagram. K-means clusters images into circuit nodes at each layer (coloured rectangles, height ∝ population). All image paths flow as dim coloured threads. The query image's path lights up in white, revealing which circuit it rode. |
| **Bottom** | Softmax probability bar chart (static) |

**Workflow:** run cells top-to-bottom.  
Cell 5 (circuit precompute) takes a few seconds — run it once and reuse.  
Edit `IMAGE_IDX` in cell 6 to explore different images.

## 0. Colab Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/jacobposchl/trainable-circuits'
REPO_DIR = '/content/model_interpretability'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn', '-q'], check=True)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 50   # MB

print(f'Repo:     {REPO_DIR}')
print(f'Data:     {DATA_DIR}')
print(f'Checkpts: {CKPT_DIR}')

## 1. Configuration

Edit **only this cell** to switch between model architectures.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  SELECT MODEL  —  uncomment one line
# ─────────────────────────────────────────────────────────────────────────────
MODEL_CONFIG = CONFIG_DIR + '/models/resnet18.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet34.yaml'
# MODEL_CONFIG = CONFIG_DIR + '/models/resnet50.yaml'

print(f'Using config: {MODEL_CONFIG}')

## 2. Load Config & Checkpoint

In [ ]:
import yaml
import torch

with open(MODEL_CONFIG) as f:
    config = yaml.safe_load(f)

config['data']['data_dir']          = DATA_DIR
config['logging']['checkpoint_dir'] = CKPT_DIR + '/' + config['experiment']['name']

checkpoint_path = config['logging']['checkpoint_dir'] + '/best.pt'
device          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Model      : {config['experiment']['name']}")
print(f"Checkpoint : {checkpoint_path}")
print(f"Device     : {device}")

## 3. Build Models

In [ ]:
from evaluation.circuit_analysis import load_checkpoint

backbone, meta_encoder, info_loss = load_checkpoint(config, checkpoint_path, device)
print(f'Backbone layers: {len(backbone.layer_dims)}')

## 4. Collect Representations

In [ ]:
from evaluation.circuit_analysis import CircuitAnalyzer
from data.cifar import get_standard_loaders

MAX_SAMPLES = 2000

_, val_loader = get_standard_loaders(
    data_dir   = config['data']['data_dir'],
    batch_size = config['data'].get('batch_size', 256),
    num_workers= config['data'].get('num_workers', 4),
    augment    = False,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
print(f'Collecting representations (max {MAX_SAMPLES} samples)...')
data = analyzer.collect_representations(max_samples=MAX_SAMPLES)

print(f"Collected {data['images'].shape[0]} images across {len(data['z_list'])} layers.")

## 5. Precompute Circuit Flow  *(run once, reuse)*

Clusters images into K circuit nodes at each layer via K-means on the
meta-encoder z-vectors.  Nodes are sorted by dominant class so images of
the same class flow through similar vertical regions.  Fast — a few seconds.

In [ ]:
from evaluation import precompute_circuit_flow

N_CLUSTERS = 6   # circuit nodes per layer — increase for finer granularity

print(f'Clustering into {N_CLUSTERS} circuit nodes per layer...')
circuit_data = precompute_circuit_flow(
    data['z_list'],
    data['labels'],
    n_clusters   = N_CLUSTERS,
    random_state = 42,
)
print(f"Done.  {circuit_data['n_layers']} layers × {circuit_data['n_clusters']} nodes.")

## 6. Preview Query Image

In [ ]:
import matplotlib.pyplot as plt
from evaluation.circuit_analysis import CIFAR10_CLASSES, denormalize

IMAGE_IDX = 42   # ← change this to explore different images

img_np = denormalize(data['images'][IMAGE_IDX]).permute(1, 2, 0).numpy()
label  = int(data['labels'][IMAGE_IDX])

fig, ax = plt.subplots(1, 1, figsize=(2.5, 2.5))
ax.imshow(img_np.clip(0, 1), interpolation='nearest')
ax.set_title(f'Index {IMAGE_IDX} — {CIFAR10_CLASSES[label]}', fontsize=10)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Animate

In [ ]:
from IPython.display import HTML
from evaluation import animate_trajectory

anim = animate_trajectory(
    image_idx    = IMAGE_IDX,
    data         = data,
    circuit_data = circuit_data,
    backbone     = backbone,
    device       = device,
    interval     = 700,   # ms per frame
    dpi          = 110,
)

HTML(anim.to_jshtml())

## 8. Save  *(optional)*

`.gif` requires Pillow &nbsp;(`pip install Pillow`)  
`.mp4` requires ffmpeg &nbsp;(`apt-get install ffmpeg`)

In [ ]:
SAVE_PATH = f'trajectory_img{IMAGE_IDX}.gif'   # or .mp4

animate_trajectory(
    image_idx    = IMAGE_IDX,
    data         = data,
    circuit_data = circuit_data,
    backbone     = backbone,
    device       = device,
    interval     = 700,
    dpi          = 120,
    save_path    = SAVE_PATH,
)

## 9. Batch  *(optional)*

Animate several images in sequence.  Each call reuses `circuit_data`
so there is no extra clustering cost.

In [ ]:
from IPython.display import display

INDICES = [0, 10, 42, 99, 200]   # ← pick any indices

for idx in INDICES:
    lbl = int(data['labels'][idx])
    print(f'\n─── Image {idx:4d}  ({CIFAR10_CLASSES[lbl]}) ───')
    a = animate_trajectory(
        image_idx    = idx,
        data         = data,
        circuit_data = circuit_data,
        backbone     = backbone,
        device       = device,
        interval     = 700,
        dpi          = 90,
    )
    display(HTML(a.to_jshtml()))